# Exploração · `bronze.faturas`

Notebook de **investigação**, fora do pipeline. Descobre os problemas de qualidade de `faturas` e a regra de limpeza de cada um, antes de escrever a Silver.

**Status: exploração concluída.** Rode com **Run all**: cada célula só depende das anteriores. Na Bronze tudo é texto, então as contas usam `try_cast`, que devolve nulo em vez de dar erro quando o valor não vira número. O `faturas.csv` usa `;` como separador.

| # | Seção | Resultado |
|---|---|---|
| 1 | Visão geral | Tudo é texto na Bronze |
| 2 | Nulos | `data_pagamento` com 113.835 nulos |
| 3 | Categorias | `bandeira_tarifaria` e `status` sem variações |
| 4 | Datas e cabeçalho repetido | 2 linhas de cabeçalho; `data_emissao` em 4 formatos |
| 5 | Duplicatas e órfãs | sem cópias; 32.285 faturas órfãs |
| 6 | Checagens | `valor_total` como moeda; datas coerentes; `9999-12-31`; 1 fatura por leitura |
| 7 | Resumo | regras propostas para a Silver |

**Sobre os números:** as seções 1 a 4 rodam sobre a tabela bruta. A partir da 4.1, `faturas` já não inclui as 2 linhas de cabeçalho repetido. Da Seção 5 em diante, `faturas_limpa` também exclui as órfãs.

In [0]:
from pyspark.sql import functions as F

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"

faturas = spark.table(f"{bronze_schema}.faturas")

## 1. Visão geral

**O que explora:** colunas, tipos e uma amostra. Na Bronze todas as colunas são `string`, e há duas colunas de auditoria (`source_file`, `ingestion_timestamp`) que não vêm do sistema de origem.

In [0]:
faturas.printSchema()
faturas.limit(5).display()
print(f"{faturas.count():,} linhas")

root
 |-- id_fatura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- data_emissao: string (nullable = true)
 |-- valor_energia: string (nullable = true)
 |-- tributos: string (nullable = true)
 |-- bandeira_tarifaria: string (nullable = true)
 |-- valor_total: string (nullable = true)
 |-- data_vencimento: string (nullable = true)
 |-- data_pagamento: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,source_file,ingestion_timestamp
FT001417237,UC129634,2023-02,2023-02-05,175.72,58.57,Verde,234.29,2023-02-17,2023-02-20,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
FT001417238,UC129634,2023-03,2023-03-13,191.99,64.00,Verde,255.99,2023-03-25,2023-03-23,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
FT001417239,UC129634,2023-04,2023-04-15,142.77,47.59,Verde,190.36,2023-04-27,2023-05-20,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
FT001417240,UC129634,2023-05,05-28-2023,142.21,47.40,Verde,189.61,2023-06-09,2023-06-26,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
FT001417241,UC129634,2023-06,2023-06-03,104.67,34.89,Verde,139.56,2023-06-15,2023-07-08,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z


2,184,623 linhas


## 2. Nulos

**O que explora:** quantas células vazias há por coluna.

**Achado:** `data_pagamento` tem **113.835 nulos reais**. A Seção 6c confirma que todos são faturas EM ABERTO (nulo de negócio), e mostra que a ausência de pagamento aparece também como `9999-12-31`.

In [0]:
faturas.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in faturas.columns]).display()

id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,source_file,ingestion_timestamp
0,0,0,0,0,0,0,0,0,113835,0,0,0


## 3. Categorias

**O que explora:** valores distintos de `bandeira_tarifaria` e `status`, procurando o mesmo valor escrito de formas diferentes (caixa, espaços).

**Achado:** sem variações de grafia. `status` tem só `PAGA` e `EM ABERTO`.

In [0]:
faturas.groupBy("bandeira_tarifaria").count().orderBy(F.desc("count")).display()
faturas.groupBy("status").count().orderBy(F.desc("count")).display()

bandeira_tarifaria,count
Verde,1546130
Amarela,273630
Vermelha P1,228042
Vermelha P2,136819
bandeira_tarifaria,2


status,count
PAGA,2009453
EM ABERTO,175168
status,2


## 4. Formatos de data e cabeçalho repetido

**O que explora:** troca cada dígito por `9` para mostrar quais formatos existem em cada coluna de data. Mais de um padrão na mesma coluna significa datas misturadas.

**Achado (tabela bruta):**
- `data_emissao` tem **4 formatos**: ISO `yyyy-MM-dd` (2.001.260, 91,6%) e três não ISO (183.361 linhas, 8,4%): `99/99/9999` (61.291), `99.99.9999` (61.252) e `99-99-9999` (60.818).
- `data_vencimento` e `mes_referencia` têm formato único (`yyyy-MM-dd` e `yyyy-MM`).
- `data_pagamento` está toda em formato ISO, mas o padrão `9999-99-99` **inclui a sentinela `9999-12-31`** (ver 6c), além de 113.835 nulos reais.
- Em todas as colunas aparecem 2 linhas cujo valor é o próprio nome da coluna (`data_emissao`, `data_vencimento`, `data_pagamento`, `mes_referencia`): é o **cabeçalho repetido** da subseção 4.1.

In [0]:
for c in ["data_emissao", "data_vencimento", "data_pagamento", "mes_referencia"]:
    print(c)
    (faturas.withColumn("padrao", F.regexp_replace(c, "[0-9]", "9"))
            .groupBy("padrao").count().orderBy(F.desc("count")).display())

data_emissao


padrao,count
9999-99-99,2001260
99/99/9999,61291
99.99.9999,61252
99-99-9999,60818
data_emissao,2


data_vencimento


padrao,count
9999-99-99,2184621
data_vencimento,2


data_pagamento


padrao,count
9999-99-99,2070786
null,113835
data_pagamento,2


mes_referencia


padrao,count
9999-99,2184621
mes_referencia,2


### 4.1 Cabeçalho repetido

**O que explora:** linhas em que o valor de uma coluna é igual ao nome da própria coluna, sinal de que o cabeçalho do CSV foi lido como dado.

**Achado:**
- **2 linhas** em `faturas`, ambas do `faturas.csv`. Sem elas, a tabela tem **2.184.621 linhas**.
- Em `unidades_consumidoras` há **1 linha** com o mesmo problema. Em `leituras`, `medidores`, `ocorrencias_rede` e `geracao_distribuida` há 0. `atendimentos` e `inspecoes` não foram testadas.
- **Decisão proposta:** remover na Silver, **antes** da deduplicação (se deduplicar primeiro, uma das duas sobra). A causa (arquivo de origem ou leitura da Bronze) deve ser conferida com quem fez a ingestão.

A última célula abaixo remove as 2 linhas: **a partir dela, `faturas` já não as inclui.**

In [0]:
# Linhas de cabeçalho repetido: o valor é igual ao nome da coluna
cab = faturas.filter(F.col("id_fatura") == "id_fatura")
print("linhas de cabeçalho repetido:", cab.count())
cab.display()

linhas de cabeçalho repetido: 2


id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,source_file,ingestion_timestamp
id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z


In [0]:
# O mesmo problema existe nas outras tabelas? (testa a primeira coluna de cada uma)
for nome in ["unidades_consumidoras", "leituras", "faturas", "medidores", "ocorrencias_rede", "geracao_distribuida"]:
    df = spark.table(f"{bronze_schema}.{nome}")
    primeira = df.columns[0]
    print(nome, "->", df.filter(F.col(primeira) == primeira).count())

unidades_consumidoras -> 1
leituras -> 0
faturas -> 2
medidores -> 0
ocorrencias_rede -> 0
geracao_distribuida -> 0


In [0]:
# A partir daqui `faturas` já não tem as linhas de cabeçalho repetido
faturas = faturas.filter(F.col("id_fatura") != "id_fatura")
print("linhas:", faturas.count(), "(esperado: 2184621)")

linhas: 2184621 (esperado: 2184621)


### 4.2 Ordem dia/mês dos formatos não ISO

**O que explora:** para cada padrão não ISO, testa os dois formatos possíveis (dia primeiro e mês primeiro). O ISO entra como controle. Gabaritos: o vencimento é sempre a emissão + 12 dias, e o mês da emissão deve bater com o `mes_referencia`.

**Achado:** o formato certo é o que converte todas as linhas e fecha o vencimento em 100%:

| Padrão | Formato | Linhas | Não converteu | Vencimento = emissão + 12 dias |
|---|---|---|---|---|
| `9999-99-99` | `yyyy-MM-dd` | 2.001.260 | 0 | 100% |
| `99-99-9999` | **`MM-dd-yyyy`** (mês primeiro) | 60.818 | 0 | 100% |
| `99.99.9999` | `dd.MM.yyyy` | 61.252 | 0 | 100% |
| `99/99/9999` | `dd/MM/yyyy` | 61.291 | 0 | 100% |

Os formatos alternativos falham: `dd-MM-yyyy` não converte 38.615 linhas, `MM.dd.yyyy` 39.039 e `MM/dd/yyyy` 39.011. É o **mesmo mapeamento de `leituras`** (hífen = mês primeiro).

**O `mes_referencia` não serve como gabarito de 100%:** mesmo no ISO, o mês da emissão só bate em 99,5% das linhas (1.990.946 de 2.001.260), provavelmente faturas emitidas no mês seguinte à competência (não verificado).

In [0]:
t4 = (faturas
        .withColumn("padrao", F.regexp_replace("data_emissao", "[0-9]", "9"))
        .withColumn("venc", F.expr("try_to_timestamp(data_vencimento, 'yyyy-MM-dd')")))

candidatos = {
    "9999-99-99": ["yyyy-MM-dd"],
    "99-99-9999": ["dd-MM-yyyy", "MM-dd-yyyy"],
    "99.99.9999": ["dd.MM.yyyy", "MM.dd.yyyy"],
    "99/99/9999": ["dd/MM/yyyy", "MM/dd/yyyy"],
}

resultados = None
for padrao, formatos in candidatos.items():
    for fmt in formatos:
        r = (t4.filter(F.col("padrao") == padrao)
              .withColumn("emissao", F.expr(f"try_to_timestamp(data_emissao, '{fmt}')"))
              .withColumn("mes_emissao", F.date_format("emissao", "yyyy-MM"))
              .withColumn("dias", F.datediff(F.col("venc"), F.col("emissao")))
              .agg(F.lit(padrao).alias("padrao"), F.lit(fmt).alias("formato_testado"),
                   F.count("*").alias("linhas"),
                   F.sum(F.col("emissao").isNull().cast("int")).alias("nao_converteu"),
                   F.sum((F.col("mes_emissao") == F.col("mes_referencia")).cast("int")).alias("mes_bate"),
                   F.sum((F.col("dias") == 12).cast("int")).alias("vencimento_12_dias")))
        resultados = r if resultados is None else resultados.unionByName(r)

resultados.display()

padrao,formato_testado,linhas,nao_converteu,mes_bate,vencimento_12_dias
9999-99-99,yyyy-MM-dd,2001260,0,1990946,2001260
99-99-9999,dd-MM-yyyy,60818,38615,1914,1787
99-99-9999,MM-dd-yyyy,60818,0,60492,60818
99.99.9999,dd.MM.yyyy,61252,0,60938,61252
99.99.9999,MM.dd.yyyy,61252,39039,2049,1908
99/99/9999,dd/MM/yyyy,61291,0,60939,61291
99/99/9999,MM/dd/yyyy,61291,39011,1922,1775


## 5. Duplicatas e faturas órfãs

**O que explora:** se `id_fatura` se repete e se a mesma UC tem mais de uma fatura no mesmo mês. Em `leituras` havia dois tipos de duplicata (cópias exatas e linhas órfãs), então se procuram os dois aqui. Também testa o formato dos ids e se a UC existe no cadastro.

**Achado:**
- **Sem cópias exatas:** 2.184.621 linhas (sem o cabeçalho repetido), com 2.184.621 `id_fatura` distintos e o mesmo número de linhas distintas.
- **32.285 faturas órfãs (1,5%):** `id_fatura` fora do padrão `FT` + 9 dígitos (ex.: `FTX0031810`), `id_uc` com 7 dígitos (ex.: `UC9305662`) e UC inexistente em `unidades_consumidoras`. As três condições coincidem nas 32.285 linhas, e as outras **2.152.336** estão todas corretas.
- **21 pares `(id_uc, mes_referencia)` duplicados**, todos formados por duas órfãs (42 linhas). Nenhuma fatura válida repete UC + mês.
- **Peso das órfãs:** somam R$ 7.644.280,57 em `valor_energia` (1,5% do total de R$ 507.506.363,30), com valor médio por fatura parecido ao das válidas (R$ 236,78 contra R$ 232,24).
- As 2.152.336 faturas válidas são o mesmo número de leituras em `limpo` de `leituras`: a Seção 6d confirma uma fatura por leitura.

**Decisão proposta:** remover as órfãs (ou mandar para quarentena). A receita total fica ~1,5% menor sem elas: registrar no dicionário de dados.

In [0]:
total = faturas.count()
ids_unicos = faturas.select("id_fatura").distinct().count()
cols = [c for c in faturas.columns if c not in ("source_file", "ingestion_timestamp")]
linhas_unicas = faturas.select(cols).distinct().count()
pares_unicos = faturas.select("id_uc", "mes_referencia").distinct().count()

print(f"linhas: {total:,}")
print(f"id_fatura distintos: {ids_unicos:,}  -> repetidos: {total - ids_unicos:,}")
print(f"linhas distintas (sem auditoria): {linhas_unicas:,}  (igual aos ids = cópias exatas)")
print(f"(id_uc, mes_referencia) distintos: {pares_unicos:,}  -> a mais: {ids_unicos - pares_unicos:,}")

linhas: 2,184,621
id_fatura distintos: 2,184,621  -> repetidos: 0
linhas distintas (sem auditoria): 2,184,621  (igual aos ids = cópias exatas)
(id_uc, mes_referencia) distintos: 2,184,600  -> a mais: 21


In [0]:
# Exemplos de UC + mês com mais de uma fatura
faturas_d = faturas.dropDuplicates(["id_fatura"])

dups = faturas_d.groupBy("id_uc", "mes_referencia").count().filter("count > 1")
print(f"pares com mais de uma fatura: {dups.count():,}")

(faturas_d.join(dups.select("id_uc", "mes_referencia"), ["id_uc", "mes_referencia"])
          .orderBy("id_uc", "mes_referencia")
          .limit(20).display())

pares com mais de uma fatura: 21


id_uc,mes_referencia,id_fatura,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,source_file,ingestion_timestamp
UC9131189,2023-10,FTX0010268,2023-10-07,98.45,32.82,Verde,"R$ 131,27",2023-10-19,2023-10-26,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9131189,2023-10,FTX0000502,2023-10-06,91.80,30.60,Verde,122.40,2023-10-18,2023-11-04,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9201920,2023-10,FTX0001488,2023-10-14,54.33,18.11,Verde,"R$ 72,44",2023-10-26,2023-10-27,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9201920,2023-10,FTX0015186,2023-10-12,78.01,26.00,Verde,104.01,2023-10-24,2023-11-05,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9236557,2022-07,FTX0026324,2022-07-04,31.50,10.50,Verde,42.00,2022-07-16,2022-07-27,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9236557,2022-07,FTX0016727,2022-07-10,39.37,13.12,Verde,"R$ 52,49",2022-07-22,2022-08-02,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9305662,2023-01,FTX0031810,2023-01-20,267.14,89.05,Verde,356.19,2023-02-01,2023-02-23,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9305662,2023-01,FTX0004374,2023-01-03,75.48,25.16,Verde,"R$ 100,64",2023-01-15,2023-01-14,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9450496,2026-01,FTX0002420,03.01.2026,584.34,194.78,Verde,"R$ 779,12",2026-01-15,2026-01-19,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
UC9450496,2026-01,FTX0011176,22/01/2026,170.87,56.96,Verde,227.83,2026-02-03,2026-01-31,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z


In [0]:
# Formato dos ids e existência da UC no cadastro
ucs = (spark.table(f"{bronze_schema}.unidades_consumidoras")
            .select("id_uc").distinct().withColumn("uc_existe", F.lit(True)))

f = (faturas
  .withColumn("fmt_fatura_ok", F.col("id_fatura").rlike("^FT[0-9]{9}$"))
  .withColumn("fmt_uc_ok", F.col("id_uc").rlike("^UC[0-9]{6}$"))
  .join(ucs, "id_uc", "left")
  .withColumn("uc_existe", F.coalesce("uc_existe", F.lit(False))))

f.groupBy("fmt_fatura_ok", "fmt_uc_ok", "uc_existe").count().display()

fmt_fatura_ok,fmt_uc_ok,uc_existe,count
true,true,true,2152336
false,false,false,32285


In [0]:
# Os pares duplicados são todos de órfãs?
pares = f.groupBy("id_uc", "mes_referencia").count().filter("count > 1")
(f.join(pares.select("id_uc", "mes_referencia"), ["id_uc", "mes_referencia"])
  .groupBy("fmt_fatura_ok", "fmt_uc_ok", "uc_existe").count().display())

fmt_fatura_ok,fmt_uc_ok,uc_existe,count
false,false,false,42


In [0]:
# Tabela sem órfãs, e quanto valem as órfãs (soma de valor_energia)
faturas_limpa = f.filter("fmt_fatura_ok AND fmt_uc_ok AND uc_existe")
print("faturas válidas:", faturas_limpa.count(),
      "| pares (uc, mês) distintos:", faturas_limpa.select("id_uc", "mes_referencia").distinct().count())

(f.withColumn("energia", F.expr("try_cast(valor_energia as double)"))
  .groupBy("uc_existe")
  .agg(F.count("*").alias("faturas"), F.round(F.sum("energia"), 2).alias("soma_valor_energia"))
  .display())

faturas válidas: 2152336 | pares (uc, mês) distintos: 2152336


uc_existe,faturas,soma_valor_energia
true,2152336,4.9986208273E8
false,32285,7644280.57


## 6. Checagens de valores, datas, status e integridade

### 6a. Os valores são números e o total bate?

**O que explora:** o formato de `valor_energia`, `tributos` e `valor_total`, e se `valor_total = valor_energia + tributos`.

**Achado (formatos, tabela bruta):** `valor_energia` e `tributos` têm um único formato (`9.9`, ponto decimal). `valor_total` tem três:

| Formato | Exemplo | Linhas |
|---|---|---|
| `9.9` | `102.57` | 1.987.999 |
| `R$ 9,9` | `R$ 100,64` | 188.571 |
| `R$ 9.9,9` | `R$ 1.234,56` (ponto de milhar) | 8.051 |

São **196.622 faturas (9,0%)** com o total escrito como moeda brasileira.

**Risco:** um `cast` simples em `valor_total` anularia essas linhas, e a receita seria subestimada em ~9%. É preciso normalizar o texto antes do cast.

**Resultado da normalização:** removendo `R$` e o ponto de milhar e trocando a vírgula decimal por ponto, **todas** as faturas válidas (2.152.336) convertem para número, e em todas `valor_total = valor_energia + tributos` (diferença zero em `decimal(12,2)`).

A célula que varre `valor_total` sem normalizar mostra **196.622** valores que não viram número: é a soma dos dois formatos de moeda (188.571 + 8.051), contada na tabela bruta, inclusive nas órfãs. Esse número é o esperado.

In [0]:
# Formatos de valor (colapsa sequências de dígitos para ver os padrões)
for c in ["valor_energia", "tributos", "valor_total"]:
    print(c)
    (faturas.withColumn("padrao", F.regexp_replace(F.regexp_replace(c, "[0-9]", "9"), "9+", "9"))
            .groupBy("padrao").count().orderBy(F.desc("count")).display())

valor_energia


padrao,count
9.9,2184621


tributos


padrao,count
9.9,2184621


valor_total


padrao,count
9.9,1987999
"R$ 9,9",188571
"R$ 9.9,9",8051


In [0]:
# Valores que não viram número sem normalizar
for c in ["valor_energia", "tributos", "valor_total"]:
    inv = faturas.filter(F.col(c).isNotNull() & F.expr(f"try_cast({c} as decimal(12,2))").isNull())
    print(f"\n{c}: {inv.count():,} valores que não viram número")
    inv.groupBy(c).count().orderBy(F.desc("count")).limit(10).display()


valor_energia: 0 valores que não viram número


valor_energia,count



tributos: 0 valores que não viram número


tributos,count



valor_total: 196,622 valores que não viram número


valor_total,count
"R$ 71,85",32
"R$ 72,16",32
"R$ 75,96",31
"R$ 78,05",29
"R$ 90,83",29
"R$ 80,65",29
"R$ 90,29",28
"R$ 73,61",28
"R$ 92,59",27
"R$ 78,48",27


In [0]:
# Normaliza valor_total (tira "R$" e ponto de milhar, vírgula vira ponto) e confere com energia + tributos
vt = F.col("valor_total")

v = faturas_limpa.withColumn("total_txt",
    F.when(vt.startswith("R$"),
           F.regexp_replace(F.regexp_replace(F.regexp_replace(vt, r"R\$[\s\u00a0]*", ""), r"\.", ""), ",", "."))
     .otherwise(vt))

v = (v.withColumn("energia", F.expr("try_cast(valor_energia as decimal(12,2))"))
      .withColumn("trib", F.expr("try_cast(tributos as decimal(12,2))"))
      .withColumn("total", F.expr("try_cast(total_txt as decimal(12,2))"))
      .withColumn("dif", F.col("energia") + F.col("trib") - F.col("total")))

print("não converteu:", v.filter("total IS NULL").count())
print("total != energia + tributos:", v.filter("dif != 0").count())
v.filter("dif != 0").select("id_fatura", "valor_energia", "tributos", "valor_total", "total_txt", "dif").limit(10).display()

não converteu: 0
total != energia + tributos: 0


id_fatura,valor_energia,tributos,valor_total,total_txt,dif


### 6b. As datas fazem sentido entre si?

**O que explora:** converte cada data com o formato certo (mapeado na 4.2) e procura emissão depois do vencimento e pagamento antes da emissão. A `data_pagamento` com a sentinela `9999-12-31` é tratada como ausente nesta checagem (`pgto_real`), senão ela passaria trivialmente.

**Achado:** com os quatro formatos de `data_emissao`, todas as emissões e vencimentos convertem (0 não convertidas), e há **0** faturas com emissão depois do vencimento e **0** com pagamento antes da emissão. Em todos os formatos, o vencimento é emissão + 12 dias (4.2).

In [0]:
p = F.regexp_replace("data_emissao", "[0-9]", "9")
fd = (faturas
  .withColumn("emissao",
      F.when(p == "9999-99-99", F.expr("try_to_timestamp(data_emissao, 'yyyy-MM-dd')"))
       .when(p == "99-99-9999", F.expr("try_to_timestamp(data_emissao, 'MM-dd-yyyy')"))
       .when(p == "99.99.9999", F.expr("try_to_timestamp(data_emissao, 'dd.MM.yyyy')"))
       .when(p == "99/99/9999", F.expr("try_to_timestamp(data_emissao, 'dd/MM/yyyy')")))
  .withColumn("venc", F.expr("try_to_timestamp(data_vencimento, 'yyyy-MM-dd')"))
  .withColumn("pgto", F.expr("try_to_timestamp(data_pagamento, 'yyyy-MM-dd')"))
  .withColumn("pgto_real", F.when(F.col("data_pagamento") != "9999-12-31", F.col("pgto"))))

print("emissão não convertida:", fd.filter("emissao IS NULL").count())
print("vencimento não convertido:", fd.filter("venc IS NULL").count())
print("emissão depois do vencimento:", fd.filter("emissao > venc").count())
print("pagamento antes da emissão (ignorando a sentinela):", fd.filter("pgto_real < emissao").count())

emissão não convertida: 0
vencimento não convertido: 0
emissão depois do vencimento: 0
pagamento antes da emissão (ignorando a sentinela): 0


### 6c. O `status` combina com a `data_pagamento`?

**O que explora:** se os nulos de `data_pagamento` correspondem a faturas em aberto e o que há de estranho na combinação `status` × data.

**Achado (tabela bruta, com órfãs):**
- `status` tem só `PAGA` e `EM ABERTO`.
- **Os 113.835 nulos de `data_pagamento` são todos EM ABERTO:** nulo de negócio, a preservar. Nenhuma fatura PAGA está sem data (2.009.453 têm data real).
- **Há 61.333 faturas EM ABERTO "com data de pagamento", e a data é `9999-12-31`**, sentinela que significa "ainda não pago" (mesmo padrão de `data_retirada` em `medidores`). A distribuição é regular, ~1.275 por mês, em todos os anos, o que indica um defeito de origem.
- Ao todo, a ausência de pagamento aparece de **duas formas**: nulo real (113.835) e `9999-12-31` (61.333). São **175.168 faturas em aberto (8,0%)**.

**Decisão proposta:** converter `9999-12-31` em nulo real e derivar a flag `em_aberto`. Evita que contas como "dias entre vencimento e pagamento" usem o ano 9999. A lista de "faturas em aberto" do atendente deve usar o `status`, e não a presença da data.

In [0]:
(faturas.withColumn("tem_data_pagamento", F.col("data_pagamento").isNotNull())
        .groupBy("status", "tem_data_pagamento").count().orderBy("status").display())

status,tem_data_pagamento,count
EM ABERTO,false,113835
EM ABERTO,true,61333
PAGA,true,2009453


In [0]:
# As faturas EM ABERTO "com data" têm a sentinela? (esperado: nulo_real 113.835, sentinela 61.333, data_real 2.009.453 em PAGA)
(faturas.withColumn("forma_pagamento",
        F.when(F.col("data_pagamento").isNull(), "nulo_real")
         .when(F.col("data_pagamento") == "9999-12-31", "sentinela_9999-12-31")
         .otherwise("data_real"))
        .groupBy("status", "forma_pagamento").count().orderBy("status").display())

status,forma_pagamento,count
EM ABERTO,nulo_real,113835
EM ABERTO,sentinela_9999-12-31,61333
PAGA,data_real,2009453


In [0]:
# Quando acontece? Por ano, e as datas mais recentes de cada status (usa `fd` da 6b)
aberto_pago = fd.filter("status = 'EM ABERTO' AND pgto IS NOT NULL")

print("EM ABERTO com data de pagamento:", aberto_pago.count())
aberto_pago.groupBy(F.substring("mes_referencia", 1, 4).alias("ano")).count().orderBy("ano").display()

fd.groupBy("status").agg(F.max("emissao").alias("ultima_emissao"),
                         F.max("pgto").alias("ultimo_pagamento"),
                         F.max("mes_referencia").alias("ultima_competencia")).display()

EM ABERTO com data de pagamento: 61333


ano,count
2022,7544
2023,15297
2024,15277
2025,15519
2026,7696


status,ultima_emissao,ultimo_pagamento,ultima_competencia
PAGA,2026-06-30T00:00:00.000Z,2026-08-06T00:00:00.000Z,2026-06
EM ABERTO,2026-06-30T00:00:00.000Z,9999-12-31T00:00:00.000Z,2026-06


### 6d. Integridade referencial

**O que explora:** se toda fatura válida tem leitura no mesmo mês e vice-versa. Compara os pares `(id_uc, mes_referencia)` de `faturas_limpa` e das leituras já sem cópias e sem órfãs.

**Achado:** **0** faturas sem leitura e **0** leituras sem fatura. Depois de remover as órfãs das duas tabelas, `faturas` e `leituras` têm exatamente os mesmos 2.152.336 pares: há **uma fatura por leitura**. Isso atende ao requisito de integridade referencial entre medição, faturamento e cadastro.

In [0]:
f_pares = faturas_limpa.select("id_uc", "mes_referencia").distinct()
l_pares = (spark.table(f"{bronze_schema}.leituras").dropDuplicates(["id_leitura"])
                .filter(F.col("id_medidor").rlike("^MD[0-9]{6}$"))
                .select("id_uc", "mes_referencia").distinct())

print("faturas sem leitura no mesmo mês:", f_pares.join(l_pares, ["id_uc", "mes_referencia"], "left_anti").count())
print("leituras sem fatura no mesmo mês:", l_pares.join(f_pares, ["id_uc", "mes_referencia"], "left_anti").count())

faturas sem leitura no mesmo mês: 0
leituras sem fatura no mesmo mês: 0


## 7. Resumo e regras propostas para a Silver

Contagens da tabela bruta, salvo indicação. A coluna "Decisão do grupo" fica pendente até alinharem com as colegas.

| Problema | Linhas | Regra proposta | Decisão do grupo |
|---|---|---|---|
| Cabeçalho repetido nos dados | 2 | remover, **antes** da deduplicação | pendente |
| Faturas órfãs (UC inexistente, ids fora do padrão) | 32.285 (1,5%; R$ 7,64 mi em `valor_energia`) | remover ou quarentena | pendente |
| `valor_total` como moeda (`R$ 9,9`, `R$ 9.9,9`) | 196.622 (9,0%) | tirar `R$` e o ponto de milhar, vírgula vira ponto, cast para `decimal(12,2)`. Validado: o total bate com energia + tributos em 100% | pendente |
| `data_emissao` em 4 formatos | 183.361 fora do ISO | converter cada padrão com o seu formato (`yyyy-MM-dd`, `dd/MM/yyyy`, `dd.MM.yyyy`, `MM-dd-yyyy`) e gravar como `date` | pendente |
| `data_pagamento` ausente em 2 formas (nulo real e `9999-12-31`) | 175.168 (8,0%) | `9999-12-31` vira nulo; flag `em_aberto`; preservar o nulo como nulo de negócio | pendente |

### Ordem das operações na Silver (a ordem muda o resultado)
1. Remover as 2 linhas de cabeçalho repetido.
2. Remover as órfãs (formato de `id_fatura` e `id_uc` e existência da UC no cadastro).
3. Normalizar `valor_total` antes do cast.
4. Converter `data_emissao` com o formato de cada padrão, e só depois conferir vencimento e pagamento.
5. Converter `9999-12-31` em nulo e derivar a flag `em_aberto`.

### Verificado e sem problema
- Sem cópias exatas.
- `status` e `bandeira_tarifaria` sem variações de grafia.
- Datas coerentes depois de converter cada formato (0 emissões depois do vencimento, 0 pagamentos antes da emissão).
- Uma fatura por leitura (0 sobras dos dois lados).

### Para levar ao grupo
- **Cabeçalho repetido** também em `unidades_consumidoras` (1 linha). Conferir `atendimentos` e `inspecoes`.
- **`9999-12-31`** aparece em `medidores` e `faturas`. Procurar em `ocorrencias_rede` e `atendimentos`. Convenção proposta: vira nulo real.
- **Órfãs com ids fora do padrão** (`MD` com 7 dígitos, `FTX`, `UC` com 7 dígitos): procurar nas demais tabelas que referenciam UC.
- Não verificado: se as órfãs de `faturas` e de `leituras` correspondem às mesmas UCs ou meses.

**A exploração de `faturas` está concluída.** Novos achados devem ir para `docs/data_quality_log.md`, e não para este notebook.